In [ ]:
data_dir = "DATA_DIRECTORY"

# Position

In [ ]:
import os
import numpy as np
import pandas as pd

from spacepy import pycdf
from spacepy.coordinates import Coords
from spacepy.time import Ticktock

In [ ]:
dfs = []

for file in os.listdir(os.path.join(data_dir, "CP_AUX_POSGSE_1M")):
    c = int(file[1])

    cdf = pycdf.CDF(os.path.join(data_dir, "CP_AUX_POSGSE_1M", file))

    pos_gse = cdf[f"sc_r_xyz_gse__C{c}_CP_AUX_POSGSE_1M"][:]
    time = cdf[f"time_tags__C{c}_CP_AUX_POSGSE_1M"][:]

    # Convert GSE to GSM
    coords_gse = Coords(data=pos_gse, dtype="GSE", carsph="car", ticks=Ticktock(time))
    coords_gsm = coords_gse.convert(returntype="GSM", returncarsph="car")

    pos_gsm = coords_gsm.data
    x_gsm = pos_gsm[:, 0]
    y_gsm = pos_gsm[:, 1]
    z_gsm = pos_gsm[:, 2]

    r = (x_gsm**2 + y_gsm**2 + z_gsm**2)**0.5
    mlt = (np.atan2(y_gsm, x_gsm) * 12 / np.pi + 12) % 24
    lat = np.degrees(np.atan2(z_gsm, (x_gsm**2 + y_gsm**2)**0.5))

    df = pd.DataFrame({"Time tag": pd.to_datetime(time), "C": c, "RAD": r, "MLT": mlt, "LAT": lat})
    df = df.set_index("Time tag")

    dfs.append(df)

df = pd.concat(dfs)

In [ ]:
df.astype(np.float32).to_hdf(os.path.join(data_dir, "position.h5"), key="df", format="table")

# OMNI

In [ ]:
import pandas as pd
from multiprocessing import Pool

In [ ]:
data = pd.read_hdf(os.path.join(data_dir, "drivers.h5"))

In [ ]:
time_windows = [
    (-0.25, 0.0),
    (-0.5, -0.25),
    (-0.75, -0.5),
    (-1.0, -0.75),
    (-1.5, -1.0),
    (-2.0, -1.5),
    (-3.0, -2.0),
    (-4.0, -3.0),
    (-6.0, -4.0),
    (-8.0, -6.0),
    (-12.0, -8.0),
    (-16.0, -12.0),
    (-24.0, -16.0),
    ]
driver_names = ["bavg", "bx_gsm", "by_gsm", "bz_gsm", "speed", "proton_density", "pdyn", "hp30", "sme"]

functions = ["current", "mean", "max", "fraction_negative"]

In [ ]:
POSSIBLE_DRIVER_LIST = [
    (driver, function, window)
    for driver in driver_names
    for function in functions
    for window in (
        [(0.0, 0.0)] if function == "current" else
        time_windows if function in ["mean", "max"] else
        time_windows if driver in ["bx_gsm", "by_gsm", "bz_gsm"] else []
    )
    ]

In [ ]:
def calculate_driver(args):
    parameter, function, left, right, data = args

    print(parameter, function, left, right)

    series = data[parameter]

    # Current value
    if function == "current":
        result = series.copy()

    else:
        # Shift the series so that a window such as
        # (-1.0, -0.75) represents 1 to 0.75 hours before
        # the current timestamp.
        width = pd.to_timedelta(right - left, unit="h")

        shifted = series.shift(
            freq=pd.to_timedelta(-left, unit="h")
        )

        rolled = shifted.rolling(window=width)

        if function == "mean":
            result = rolled.mean()

        elif function == "max":
            result = rolled.max()

        elif function == "fraction_negative":
            result = rolled.apply(
                lambda x: (x < 0).mean(),
                raw=True,
            )

        # Align the end of the rolling window with the
        # requested right-hand boundary.
        result = result.shift(freq=-width)

    name = f"{parameter} - {function} {left} {right}"

    return name, result

args = [
    (parameter, function, left, right, data)
    for parameter, function, (left, right) in POSSIBLE_DRIVER_LIST
]

with Pool(processes=16) as pool:
    results = pool.map(calculate_driver, args)

drivers = dict(results)

drivers = pd.DataFrame(drivers, index=data.index)

In [ ]:
drivers.astype(np.float32).to_hdf(os.path.join(data_dir, "driver_history.h5"), key="df", format="table")

# Dipole tilt

In [ ]:
from geopack import geopack

In [ ]:
# Dipole tilt
time_list = drivers.index

def calculate_tilt(time):
    tilt = geopack.recalc(pd.to_timedelta(time - datetime(1970, 1, 1, tzinfo=timezone.utc)).total_seconds())
    return tilt

pool = Pool(64)
tilts = pool.map(calculate_tilt, time_list)

In [ ]:
df = pd.DataFrame({"Dipole tilt": tilts}, index=drivers.index)

In [ ]:
df.astype(np.float32).to_hdf(os.path.join(data_dir, "tilts.h5"), key="df", format="table")

# L*

In [ ]:
from datetime import datetime

from spacepy import irbempy
from spacepy.time import Ticktock
from spacepy.coordinates import Coords

In [ ]:
pos = pd.read_hdf(os.path.join(data_dir, "position.h5"))
pos.index = pd.to_datetime(pos.index, utc=True)

In [ ]:
hp30 = pd.read_hdf(os.path.join(data_dir, "drivers.h5"))["hp30"]
pos = pos.merge(hp30, on='Time tag')

In [ ]:
r_e = 6378
x = -pos['RAD'] * np.cos(pos['MLT'] * np.pi / 12) * np.cos(np.deg2rad(pos['LAT'])) / r_e
y = -pos['RAD'] * np.sin(pos['MLT'] * np.pi / 12) * np.cos(np.deg2rad(pos['LAT'])) / r_e
z = pos['RAD'] * np.sin(np.deg2rad(pos['LAT'])) / r_e

In [ ]:
ticks = Ticktock(np.array(pos.index, dtype=datetime))
coords = Coords(np.array([x, y, z]).T, 'GSM', 'car', use_irbem=True)

In [ ]:
omni_parameters = ['Kp', 'Dst', 'dens', 'velo', 'Pdyn', 'ByIMF', 'BzIMF', 'G1',
                   'G2', 'G3', 'W1', 'W2', 'W3', 'W4', 'W5', 'W6', 'AL']

kp = np.array(pos['hp30'])
omni_vals = {omni_parameters[i]: np.zeros_like(kp) for i in range(len(omni_parameters))}
omni_vals['Kp'] = kp

In [ ]:
lstar = irbempy.get_Lstar(ticks=ticks, loci=coords, extMag='T89', omnivals=omni_vals)['Lstar']

In [ ]:
pos['L*'] = lstar.ravel()

In [ ]:
df = pos[['L*', 'C']]

In [ ]:
df.astype(np.float32).to_hdf(os.path.join(data_dir, "l_star.h5"), key="df", format="table")

# PEACE

In [ ]:
import os
import numpy as np
import pandas as pd

from spacepy.pycdf import CDF
from datetime import datetime, timezone
from scipy.interpolate import PchipInterpolator

In [ ]:
# Data are available on request
penetrating_radiation_flag_path = None

In [ ]:
dfs = []

for file in os.listdir(os.path.join(data_dir, "CP_PEA_PITCH_SPIN_DPFlux")):
    print(file)

    c = int(file[1])

    cdf = CDF(os.path.join(data_dir, "CP_PEA_PITCH_SPIN_DPFlux", file))

    timestamp = file.split("__")[1][:15]
    dt = datetime.strptime(timestamp, "%Y%m%d_%H%M%S")
    year = dt.year
    day_of_year = dt.timetuple().tm_yday
    
    time = pd.to_datetime(cdf[f"time_tags__C{c}_CP_PEA_PITCH_SPIN_DPFlux"][:])

    if len(time) <= 1:
        continue

    flux = cdf[f"Data__C{c}_CP_PEA_PITCH_SPIN_DPFlux"][:]
    energy = cdf[f"Sweep_Energy__C{c}_CP_PEA_PITCH_SPIN_DPFlux"][:]
    quality = cdf[f"Status_Quality__C{c}_CP_PEA_PITCH_SPIN_DPFlux"][:]

    # Mark outliers and missing
    flux[(flux < 0) | (flux > 1e20)] = np.nan

    # Resample to 9 pitch-angle bins
    pa_12 = np.arange(7.5, 180, 15)
    pa_9 = np.arange(10, 180, 20)
    i = np.searchsorted(pa_12, pa_9) - 1
    w = (pa_9 - pa_12[i]) / (pa_12[i + 1] - pa_12[i])
    flux = np.where(
        ~np.isnan(flux[:, i, :]) & ~np.isnan(flux[:, i + 1, :]),
        flux[:, i, :] + w[None, :, None] * (flux[:, i + 1, :] - flux[:, i, :]),
        np.nan
        )

    df = pd.DataFrame({f"Flux {10+20*i}": flux[:, i, :].ravel() for i in range(9)}, index=np.repeat(time, flux.shape[2]))
    df.index.name = "Time tag"
    df["Energy"] = energy.ravel()
    df["Quality"] = np.repeat(quality, flux.shape[2])

    df = df[~df.iloc[:, :-2].isna().all(axis=1)]

    # Select energy channels
    channels = np.array([(0.08, 0.10), (0.20, 0.25), (0.49, 0.60), (1.2, 1.46), (2.9, 3.5), (7.0, 8.5), (16.5, 20.7)]) * 1000

    df_chs = []
    for ch, e_lims in enumerate(channels):
        df_ch = df[(df["Energy"] >= e_lims[0]) & (df["Energy"] < e_lims[1])]
        df_ch["Energy channel"] = ch
        df_chs.append(df_ch)

    df = pd.concat(df_chs)

    df["C"] = c

    if year < 2001:
        continue

    # Penetrating radiation flag
    if penetrating_radiation_flag_path is not None:
        if os.path.exists(f"{penetrating_radiation_flag_path}/C{c}/{year}/CL{c}_{year}_{str(day_of_year).zfill(3)}_PEN_RAD_FLAG.time"):

            pr_file = f"{penetrating_radiation_flag_path}/C{c}/{year}/CL{c}_{year}_{str(day_of_year).zfill(3)}_PEN_RAD_FLAG.time"
            pr = pd.read_csv(pr_file, sep=r'\s+', names=['C', 'Time tag', 'Penetrating Radiation Flag'])
            pr['Time tag'] = pd.to_datetime(pr['Time tag'])
            pr = pr.set_index('Time tag')
            pr.index = pr.index.tz_localize(None)
            if year <= 2012:
                pr["Not bad"] = pr["Penetrating Radiation Flag"] <= 2
            else:
                pr["Not bad"] = pr["Penetrating Radiation Flag"] <= 1

            df = df.sort_values(["Time tag", "Energy channel", "Energy"])
            pr.index = pd.to_datetime(pr.index).astype("datetime64[ns]")
            df.index = pd.to_datetime(df.index).astype("datetime64[ns]")
            pr = pr.sort_index()
            df = pd.merge_asof(df, pr["Not bad"], on="Time tag", direction="nearest")
            df = df.set_index("Time tag")

            df = df[df["Not bad"] != False].drop(columns="Not bad")

    df = df[df["Quality"] >= 2].drop(columns="Quality")

    if len(df) == 0:
        continue

    df = pd.concat([df_ch.resample("1min").mean() for _, df_ch in df.groupby("Energy channel")])

    df = df[~df.isna().all(axis=1)]

    df["Energy channel"] = df["Energy channel"].astype(int)
    df["C"] = df["C"].astype(int)

    dfs.append(df)

df = pd.concat(dfs)
df = df.sort_values(["Time tag", "C", "Energy channel", "Energy"])

In [ ]:
df.astype(np.float32).to_hdf(os.path.join(data_dir, "peace.h5"), key="df", format="table")

# RAPID

In [ ]:
# E3DD
dfs = []

for file in sorted(os.listdir(os.path.join(data_dir, "CP_RAP_PAD_E3DD"))):
    print(file)

    c = int(file[1])
    
    cdf = CDF(os.path.join(data_dir, "CP_RAP_PAD_E3DD", file))

    time = cdf[f"Time_tags__C{c}_CP_RAP_PAD_E3DD"][:]
    if len(time) <= 1:
        continue
    
    flux = cdf[f"PAD_Electron_Dif_flux__C{c}_CP_RAP_PAD_E3DD"][:, 0:3:2, :]
    b_stability = cdf[f"B_Stability__C{c}_CP_RAP_PAD_E3DD"][:]

    # Mark outliers and missing
    flux[(flux < 0) | (flux > 1e20)] = np.nan

    df = pd.DataFrame({f"Flux {10+20*i}": flux[:, :, i].ravel() for i in range(9)}, index=np.repeat(time, flux.shape[1]))
    df.index.name = "Time tag"
    df["Energy"] = [44492.69, 80221.25] * len(time)
    df["Energy channel"] = [7, 8] * len(time)
    df["C"] = c
    df["B stability"] = np.repeat(b_stability, flux.shape[1])

    df = df[df["B stability"] >= 0.7]
    df = df.drop(columns="B stability")

    if len(df) == 0:
        continue

    df = pd.concat([df_ch.resample("1min").mean() for _, df_ch in df.groupby("Energy channel")])

    df = df[~df.isna().all(axis=1)]

    df["Energy channel"] = df["Energy channel"].astype(int)
    df["C"] = df["C"].astype(int)

    dfs.append(df)

df = pd.concat(dfs)
df = df.sort_values(["Time tag", "C", "Energy channel", "Energy"])

In [ ]:
df.astype(np.float32).to_hdf(os.path.join(data_dir, "rapid_e3dd.h5"), key="df", format="table")

In [ ]:
# L3DD
dfs = []

for file in sorted(os.listdir(os.path.join(data_dir, "CP_RAP_PAD_L3DD"))):
    print(file)

    c = int(file[1])

    cdf = CDF(os.path.join(data_dir, "CP_RAP_PAD_L3DD", file))

    time = cdf[f"Time_tags__C{c}_CP_RAP_PAD_L3DD"][:]
    if len(time) <= 1:
        continue
    
    flux = cdf[f"PAD_Electron_L_Dif_flux__C{c}_CP_RAP_PAD_L3DD"][:]
    b_stability = cdf[f"B_Stability__C{c}_CP_RAP_PAD_L3DD"][:]

    # Mark outliers and missing
    flux[(flux < 0) | (flux > 1e20)] = np.nan

    df = pd.DataFrame({f"Flux {10+20*i}": flux[:, :, i].ravel() for i in range(9)}, index=np.repeat(time, flux.shape[1]))
    df.index.name = "Time tag"
    df["Energy"] = [44492.69, 80221.25] * len(time)
    df["Energy channel"] = [7, 8] * len(time)
    df["C"] = c
    df["B stability"] = np.repeat(b_stability, flux.shape[1])

    df = df[df["B stability"] >= 0.7]
    df = df.drop(columns="B stability")

    if len(df) == 0:
        continue

    df = pd.concat([df_ch.resample("1min").mean() for _, df_ch in df.groupby("Energy channel")])

    df = df[~df.isna().all(axis=1)]

    df["Energy channel"] = df["Energy channel"].astype(int)
    df["C"] = df["C"].astype(int)

    dfs.append(df)

df = pd.concat(dfs)
df = df.sort_values(["Time tag", "C", "Energy channel", "Energy"])

In [ ]:
df.astype(np.float32).to_hdf(os.path.join(data_dir, "rapid_l3dd.h5"), key="df", format="table")

In [ ]:
# Background
dfs = []

def omnidirectional(x):
    x = pd.DataFrame(x).interpolate().bfill().ffill().to_numpy().ravel()
    sina = np.sin(np.deg2rad(np.arange(10, 181, 20)))
    return np.sum(np.array(x) * sina) / np.sum(sina)

for file in sorted(os.listdir(os.path.join(data_dir, "CP_RAP_IES_BG"))):
    print(file)

    c = int(file[1])

    cdf = CDF(os.path.join(data_dir, "CP_RAP_IES_BG", file))

    flux = cdf[f"IES_BG_Dif_flux__C{c}_CP_RAP_IES_BG"][:, 0:3:2, :]
    time = cdf[f"Time_tags__C{c}_CP_RAP_IES_BG"][:]

    # Mark outliers and missing
    flux[(flux < 0) | (flux > 1e20)] = np.nan

    flux = np.apply_along_axis(omnidirectional, axis=2, arr=flux)

    df = pd.DataFrame({"Background": flux.ravel(), "Energy channel": [7, 8] * len(flux), "C": c},
        index=pd.to_datetime(np.repeat(time, 2)))
    df.index.name = "Time tag"

    df = pd.concat(df_e.resample("1min").interpolate() for _, df_e in df.groupby("Energy channel"))

    dfs.append(df)

df_bg = pd.concat(dfs)
df_bg = df.sort_values(["Time tag", "C", "Energy channel"])

In [ ]:
df = pd.concat([
    pd.read_hdf(os.path.join(data_dir, "rapid_e3dd.h5"))[:datetime(2024, 1, 1)],
    pd.read_hdf(os.path.join(data_dir, "rapid_l3dd.h5"))[datetime(2024, 1, 1):]
    ])

In [ ]:
# Subtract the background
df = df.merge(df_bg, on=['Time tag', 'Energy channel', 'C'])

for angle in np.arange(10, 180, 20):
    df[f'Flux {angle}'] = (df[f'Flux {angle}'] - df['Background']).clip(lower=0)

df = df.drop(columns="Background")

In [ ]:
# Correct for contamination
l_star = pd.read_hdf(os.path.join(data_dir, "l_star.h5"))

contamination_perc = [
    [2.63, 2.87, 19.59, 31.41, 9.27, 1.33, 0.44, 0.26, 0.17],
    [12.29, 12.04, 43.18, 48.7, 20.2, 6.15, 2.68, 1.61, 1.06],
    ]
contamination_perc = np.array(contamination_perc)

splines = [PchipInterpolator(range(1, 30), [*contamination_perc[ch, :], *np.zeros(20)]) for ch in range(2)]

def correction_factor(ch, l_star):
    if np.isnan(l_star):
        return 1

    return 1 - splines[ch](l_star) / 100

df.index = pd.to_datetime(df.index, utc=True)
df = df.merge(l_star, on=['Time tag', 'C'])

df['Correction factors'] = [correction_factor(ch - 7, l_star) for ch, l_star in zip(df['Energy channel'], df['L*'])]

for angle in np.arange(10, 180, 20):
    df[f'Flux {angle}'] = df[f'Flux {angle}'] * df['Correction factors']

df = df.drop(columns=["L*", "Correction factors"])

In [ ]:
df.astype(np.float32).to_hdf(os.path.join(data_dir, "rapid.h5"), key="df", format="table")